# Mondkrater: echter Lauf auf Kaggle (NASA-IBM Lunar Foundation Model, WAC)

Ziel: NASAs veröffentlichten Wert **0,2581 mAP** (WAC-Krater, Patch 8, LoRA) mit 3 Wiederholungen nachmessen.
Einstellung: Patch 8, fp16 (`16-mixed`), Batch 1 × Gradient-Akkumulation 4, `deterministic=False`, 30 Epochen, Seeds 46/47/48,
offizielle Aufteilung 800 Training / 100 Validierung / 100 Test, Lernwerte aus der veröffentlichten `WAC_config.yaml`.

**Vor dem Start (einmalig):** Konto mit bestätigter Telefonnummer. Rechts unter *Settings*: **Accelerator: GPU T4 x2**, **Internet: an**.

**Ablauf**
1. **Probelauf:** In Zelle 1 steht `MODE = "smoke"`. Oben *Run All* im Editor (ca. 15 Minuten). Die letzte Zelle muss „SMOKE OK“ zeigen.
2. **Echter Lauf:** In Zelle 1 `MODE = "full"` setzen. Dann *Save Version* → **Save & Run All (Commit)** → *Save*. Der Lauf arbeitet im Hintergrund, der Browser darf zu. Nach höchstens etwa 11,5 Stunden ist die Version fertig.
3. **Fortsetzen:** Steht in der letzten Ausgabe „noch nicht fertig“: Notebook öffnen → rechts *Input* → *Add Input* → *Your Work* → **dieses Notebook** (neueste Version) hinzufügen. Dann wieder *Save & Run All*. Das Notebook findet den alten Stand selbst und macht weiter. Vor jedem weiteren Durchgang die Eingabe auf die **neueste** Version umstellen.

Daten: Modell und Grundmodell (Apache-2.0) und der Benchmark `Sombench-WAC-Crater-Detection` (CC BY 4.0) werden zur Laufzeit von Hugging Face geladen, mit festen Versionen. Die Bilder liegen nur im Zwischenspeicher und landen **nicht** in der Ausgabe. Heruntergeladene Kacheln nicht festschreiben.

In [ ]:
# 1) Einstellungen
import os, time, json, subprocess, sys
MODE = "full"   # "smoke" = Probelauf (~15 min) | "full" = echter Lauf (Save & Run All)
START = time.time()
os.environ["MPLBACKEND"] = "Agg"   # Kaggle/Colab setzen ein Backend, das nur im Kernel existiert
SESSION_HOURS = 11.0               # Training stoppt sauber vorher; Kaggle bricht eine Version nach 12 h ab
DEADLINE = START + SESSION_HOURS * 3600
PY = "/tmp/py312/bin/python"; CODE = "/tmp/realrun"; DATA = "/tmp/lunar_data"
WORK = "/kaggle/working/lunar_real" if MODE == "full" else "/kaggle/working/lunar_smoke"
SEEDS = "46,47,48" if MODE == "full" else "46,47"
os.makedirs(CODE, exist_ok=True); os.makedirs(WORK, exist_ok=True)
ENV = {**os.environ, "MPLBACKEND": "Agg"}
print("Modus:", MODE, "| Ausgabe:", WORK, "| Kernel-Python:", sys.version.split()[0])

### 2) Skripte schreiben

In [ ]:
%%writefile /tmp/realrun/setup_env.sh
#!/usr/bin/env bash
# Separate Python 3.12 environment for the NASA repo (it requires Python >=3.11,<3.13;
# Kaggle's and Colab's kernels are 3.13). The notebook kernel itself is not touched, so no restart is needed.
# Usage: bash setup_env.sh [--force-torch]   (--force-torch: reinstall the CUDA torch wheels)
# Env overrides (for testing elsewhere): VENV, TORCH_INDEX, REQUIRE_CUDA=0 (accept a CPU torch build)
set -euo pipefail
export MPLBACKEND=Agg  # Colab exports an inline backend that crashes matplotlib outside the kernel
VENV="${VENV:-/tmp/py312}"  # outside /kaggle/working, so it is not saved as output
# cu126 wheels: contain kernels for the T4 (sm_75) and run on every driver >= 525, incl. newer 5xx drivers.
TORCH_INDEX="${TORCH_INDEX:-https://download.pytorch.org/whl/cu126}"
REQUIRE_CUDA="${REQUIRE_CUDA:-1}"
UV_VERSION="${UV_VERSION:-0.13.8}"
REPO_REF="${REPO_REF:-d54c67aad513cb9daca444afa425cfb278e4fbf8}"  # pinned upstream commit (20.09.2026)
REPO="git+https://github.com/NASA-IMPACT/NASA-IBM-Lunar-Foundation-Model.git@$REPO_REF"
PY="$VENV/bin/python"

status() {  # always printed, also on the "already done" path
  "$PY" - <<'PY'
import sys, torch, importlib.metadata as m
print("Python", sys.version.split()[0], "| torch", torch.__version__, "| CUDA-Build", torch.version.cuda,
      "| terratorch", m.version("terratorch"), "| GPU sichtbar:", torch.cuda.is_available())
PY
}
repo_ok()  { [ -x "$PY" ] && "$PY" -c "import ni_lfm, terratorch_integration" 2>/dev/null; }
torch_ok() {  # importable torch; with REQUIRE_CUDA=1 it must be a CUDA build (CPU-only wheels have torch.version.cuda None)
  [ -x "$PY" ] && "$PY" -c "import torch, torchvision, sys; sys.exit(0 if ($REQUIRE_CUDA == 0 or torch.version.cuda) else 1)" 2>/dev/null; }
install_torch() {
  echo "Installiere PyTorch-Räder von $TORCH_INDEX ..."
  uv pip install --quiet --python "$PY" --reinstall-package torch --reinstall-package torchvision \
    --index-url "$TORCH_INDEX" "torch==2.12.1" "torchvision==0.27.1"
  # Pin these exact builds so the repo install cannot swap them for other CUDA builds.
  "$PY" -c "import torch, torchvision; print(f'torch=={torch.__version__}\ntorchvision=={torchvision.__version__}')" > "$VENV/pins.txt"
}
install_repo() {
  uv pip install --quiet --python "$PY" --constraint "$VENV/pins.txt" \
    --index-url https://pypi.org/simple --extra-index-url "$TORCH_INDEX" --index-strategy unsafe-best-match \
    "$REPO" "terratorch[peft]==1.2.13"
}

FORCE=0; [ "${1:-}" = "--force-torch" ] && FORCE=1
if [ $FORCE = 0 ] && repo_ok && torch_ok; then
  echo "Umgebung $VENV ist schon fertig."; status; exit 0; fi
pip -q install "uv==${UV_VERSION}"
[ -x "$PY" ] || uv venv --quiet --python 3.12 "$VENV"
if [ $FORCE = 1 ] || ! torch_ok; then install_torch; fi
[ -f "$VENV/pins.txt" ] || install_torch
repo_ok || install_repo
torch_ok || { echo "FEHLER: torch in $VENV ist kein CUDA-Build."; status; exit 1; }
status


In [ ]:
%%writefile /tmp/realrun/fetch_data.py
"""Download everything at runtime from Hugging Face, pinned to fixed revisions.
Nothing from the Sombench dataset (CC BY 4.0) is written to the output folder except the
list of tile names in manifest.json. Data goes to a scratch folder that Kaggle does not save.

Usage: python fetch_data.py DATA_DIR MANIFEST_JSON [--subset N_TRAIN,N_VAL,N_TEST] [--seeds 46,47,48]
  full run: all official splits (800 / 100 / 100 tiles)
  --subset: smoke test; per split the first N tiles sorted by image id (deterministic)"""
import os
os.environ["MPLBACKEND"] = "Agg"
os.environ.setdefault("HF_HUB_DOWNLOAD_TIMEOUT", "120")   # default 10 s is too short when the Hub is busy
os.environ.setdefault("HF_HUB_ETAG_TIMEOUT", "60")
import sys, json, hashlib, argparse, time, random
from concurrent.futures import ThreadPoolExecutor
from huggingface_hub import hf_hub_download as _hf_hub_download

# Hugging Face sometimes answers 429 "Too Many Requests / maximum queue size reached" or 5xx when busy.
# Every file is fetched on its own (no repo tree listing) and retried with exponential backoff.
RETRY_TRIES, RETRY_BASE, RETRY_MAX = 10, 30, 300
def _status(e):
    r = getattr(e, "response", None)
    return getattr(r, "status_code", None)
def hf_hub_download(*args, **kw):
    for attempt in range(1, RETRY_TRIES + 1):
        try:
            return _hf_hub_download(*args, **kw)
        except Exception as e:
            code = _status(e)
            transient = code in (408, 425, 429, 500, 502, 503, 504) or code is None and any(
                w in type(e).__name__ for w in ("Timeout", "Connection", "ChunkedEncoding", "Protocol"))
            if not transient or attempt == RETRY_TRIES:
                raise RuntimeError(f"Hugging-Face-Download endgueltig fehlgeschlagen nach {attempt} Versuchen "
                                   f"({args[0]} / {args[1] if len(args) > 1 else kw.get('filename')}): HTTP {code} {type(e).__name__}: {str(e)[:300]}") from e
            wait = min(RETRY_MAX, RETRY_BASE * 2 ** (attempt - 1)) * random.uniform(0.8, 1.2)
            print(f"  HF HTTP {code} ({type(e).__name__}) bei {args[1] if len(args) > 1 else kw.get('filename')}, "
                  f"Versuch {attempt}/{RETRY_TRIES}, warte {wait:.0f}s", flush=True)
            time.sleep(wait)

DATASET = ("nasa-ibm-ai4science/Sombench-WAC-Crater-Detection", "20f800becb64a0c7ad314652a683f97f0281fe67")
BACKBONE = ("nasa-ibm-ai4science/NASA-IBM-Lunar-Foundation-Model", "b657bdfe6fc4c0445d918f77a2d7b919805f3bce")
CRATER = ("nasa-ibm-ai4science/Crater-Detection-NASA-IBM-Lunar-Foundation-Model", "f09ebeae2099c28f09bf4060f56d3e327a6192de")
UPSTREAM_CODE = "d54c67aad513cb9daca444afa425cfb278e4fbf8"

p = argparse.ArgumentParser()
p.add_argument("data_dir"); p.add_argument("manifest")
p.add_argument("--subset", default=""); p.add_argument("--seeds", default="46,47,48")
a = p.parse_args()
d = a.data_dir; os.makedirs(d, exist_ok=True); t0 = time.time()
sha = lambda f: hashlib.sha256(open(f, "rb").read()).hexdigest()

cfg = hf_hub_download(CRATER[0], "WAC_config.yaml", revision=CRATER[1], local_dir=f"{d}/model")
for f in ["backbone/config.yaml", "backbone/checkpoint.pt"]:   # the only files in backbone/ at this revision
    hf_hub_download(BACKBONE[0], f, revision=BACKBONE[1], local_dir=f"{d}/model")
print("backbone ready", flush=True)
sub = dict(zip(["train", "val", "test"], map(int, a.subset.split(",")))) if a.subset else {}
splits = {}
os.makedirs(f"{d}/ann", exist_ok=True)
for split in ["train", "val", "test"]:
    src = hf_hub_download(DATASET[0], f"{split}.json", repo_type="dataset", revision=DATASET[1], local_dir=f"{d}/hf")
    full = json.load(open(src))
    imgs = sorted(full["images"], key=lambda i: i["id"])
    if split in sub: imgs = imgs[:sub[split]]
    ids = {i["id"] for i in imgs}
    out = {k: v for k, v in full.items() if k not in ("images", "annotations")}
    out["images"] = imgs; out["annotations"] = [x for x in full["annotations"] if x["image_id"] in ids]
    json.dump(out, open(f"{d}/ann/{split}.json", "w"))
    stems = [os.path.splitext(os.path.basename(i["file_name"]))[0] for i in imgs]
    splits[split] = {"source_json_sha256": sha(src), "n_tiles": len(stems), "n_annotations": len(out["annotations"]), "tiles": stems}
    print(split, len(stems), "tiles", len(out["annotations"]), "annotations", flush=True)

all_stems = [s for v in splits.values() for s in v["tiles"]]
files = [f"images_tiff/{s}.tif" for s in all_stems] + ["metadata.parquet"]
fetch = lambda f: hf_hub_download(DATASET[0], f, repo_type="dataset", revision=DATASET[1], local_dir=f"{d}/hf")
with ThreadPoolExecutor(4) as ex:   # 4 parallel downloads: fast enough, gentle on the Hub
    for n, _ in enumerate(ex.map(fetch, files), 1):
        if n % 100 == 0 or n == len(files): print(f"  tiles {n}/{len(files)}", flush=True)
os.makedirs(f"{d}/hf/images_tiff", exist_ok=True)
missing = [s for s in all_stems if not os.path.exists(f"{d}/hf/images_tiff/{s}.tif")]
assert not missing, f"{len(missing)} tiles missing, e.g. {missing[:3]}"

manifest = {
    "created_utc": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
    "mode": "subset " + a.subset if a.subset else "full official splits",
    "seeds": [int(s) for s in a.seeds.split(",")],
    "revisions": {"dataset": DATASET, "backbone": BACKBONE, "crater_model_config": CRATER, "upstream_code_commit": UPSTREAM_CODE},
    "config_sha256": sha(cfg),
    "license_note": "Sombench-WAC-Crater-Detection is CC BY 4.0 and is downloaded at runtime only; this file lists tile names, no image data.",
    "splits": splits,
}
os.makedirs(os.path.dirname(os.path.abspath(a.manifest)), exist_ok=True)
json.dump(manifest, open(a.manifest, "w"), indent=1)
print(f"data ready in {time.time()-t0:.0f}s:", d, "| manifest:", a.manifest)


In [ ]:
%%writefile /tmp/realrun/train_seed.py
"""Train + test one seed of the WAC crater LoRA fine-tune, with checkpoint every epoch and auto-resume.
Writes into RUN_DIR: ckpt/last.ckpt, ckpt/best.ckpt, best.json, metrics.jsonl (one line per epoch),
status.json (progress), result.json (final test metrics; its presence means 'seed done').

Usage: python train_seed.py --seed 46 --config WAC_config.yaml --backbone-dir DIR --data-dir DIR --ann-dir DIR
       --run-dir DIR [--epochs 30 --batch 1 --accum 4 --precision 16-mixed --patch 8 --deterministic 0
       --limit-train 0 --limit-eval 0 --deadline UNIX_TS --num-workers 2]"""
import os
os.environ["MPLBACKEND"] = "Agg"
os.environ.setdefault("CUBLAS_WORKSPACE_CONFIG", ":4096:8")
import sys, glob, json, time, shutil, argparse, yaml, torch, lightning.pytorch as pl
from lightning.pytorch.callbacks import Callback
import terratorch_integration
from terratorch_integration import LunarObjectDetectionTask, DeterministicAugmentation
from terratorch_integration.data_adapter import LunarWACCraterDataModule
from terratorch_integration.data_utils import D4DetectionTransform

p = argparse.ArgumentParser()
for k, v in [("--seed", int), ("--config", str), ("--backbone-dir", str), ("--data-dir", str), ("--ann-dir", str), ("--run-dir", str)]:
    p.add_argument(k, type=v, required=True)
p.add_argument("--epochs", type=int, default=30); p.add_argument("--batch", type=int, default=1)
p.add_argument("--accum", type=int, default=4); p.add_argument("--precision", default="16-mixed")
p.add_argument("--patch", type=int, default=8); p.add_argument("--deterministic", type=int, default=0)
p.add_argument("--limit-train", type=int, default=0); p.add_argument("--limit-eval", type=int, default=0)
p.add_argument("--deadline", type=float, default=0); p.add_argument("--num-workers", type=int, default=2)
a = p.parse_args()
R = a.run_dir; os.makedirs(f"{R}/ckpt", exist_ok=True)
now = lambda: time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime())
def write_json(path, obj):  # atomic: a kill never leaves a half-written file
    tmp = path + ".tmp"; json.dump(obj, open(tmp, "w"), indent=1); os.replace(tmp, path)
if os.path.exists(f"{R}/result.json"):
    print(f"seed {a.seed}: already done ->", json.load(open(f"{R}/result.json"))["test_map"]); sys.exit(0)

cuda = torch.cuda.is_available()
if a.precision == "amp":
    a.precision = "bf16-mixed" if cuda and torch.cuda.get_device_capability(0)[0] >= 8 else ("16-mixed" if cuda else "bf16-mixed")
pl.seed_everything(a.seed, workers=True)
cfg = yaml.safe_load(open(a.config)); m = cfg["model"]["init_args"]; d = cfg["data"]["init_args"]
m["model_args"]["backbone_checkpoint_path"] = f"{a.backbone_dir}/checkpoint.pt"
m["model_args"]["backbone_cfg"] = f"{a.backbone_dir}/config.yaml"
if a.patch != 8:
    m["model_args"]["backbone_patch_size"] = a.patch
    for n in m["model_args"]["necks"]:
        if n["name"] == "ReshapeTokensToImage": n["h"] = d["image_size"] // a.patch
model = LunarObjectDetectionTask(**m)
dm = LunarWACCraterDataModule(
    data_dir=a.data_dir, train_annotations_file=f"{a.ann_dir}/train.json", val_annotations_file=f"{a.ann_dir}/val.json",
    test_annotations_file=f"{a.ann_dir}/test.json", batch_size=a.batch, num_workers=a.num_workers,
    image_size=d["image_size"], min_diameter_px=d["min_diameter_px"], drop_no_crater_images=d["drop_no_crater_images"],
    wac_norm_means=d["wac_norm_means"], wac_norm_stds=d["wac_norm_stds"],
    train_transforms=D4DetectionTransform(image_keys=["image", "vis"], p=1.0))

class EpochLog(Callback):
    """One JSON line per finished epoch; also stops cleanly before an epoch that would not fit the deadline."""
    def __init__(s): s.durations = []
    def on_train_epoch_start(s, trainer, pl_module):
        s.t0 = time.time(); s.losses = []; s.nan_losses = 0; s.skipped = 0
        if cuda: torch.cuda.reset_peak_memory_stats()
    def on_train_batch_end(s, trainer, pl_module, outputs, batch, batch_idx):
        v = trainer.callback_metrics.get("train_loss")
        if v is None or not hasattr(s, "losses"): return
        if torch.isfinite(v): s.losses.append(float(v))
        else: s.nan_losses += 1
    def on_before_optimizer_step(s, trainer, pl_module, optimizer):
        # Guard: a single non-finite gradient (possible with mixed precision) must not poison the weights.
        # With fp16 the GradScaler already skips such steps; this also covers bf16/fp32.
        grads = [q.grad for g in optimizer.param_groups for q in g["params"] if q.grad is not None]
        if grads and not all(torch.isfinite(g).all() for g in grads):
            for g in grads: g.zero_()
            s.skipped += 1
    def on_train_epoch_end(s, trainer, pl_module):  # runs after this epoch's validation
        dur = time.time() - getattr(s, "t0", time.time()); s.durations.append(dur); cm = trainer.callback_metrics
        rec = {"seed": a.seed, "epoch": trainer.current_epoch + 1, "utc": now(),
               "train_loss": round(sum(s.losses) / len(s.losses), 5) if s.losses else None,
               "val_map": round(float(cm["val_map"]), 5) if "val_map" in cm else None,
               "val_map_50": round(float(cm["val_map_50"]), 5) if "val_map_50" in cm else None,
               "seconds": round(dur, 1),
               "peak_vram_GB": round(torch.cuda.max_memory_allocated() / 1e9, 2) if cuda else None,
               "lr": trainer.optimizers[0].param_groups[0]["lr"],
               "nan_loss_batches": s.nan_losses, "skipped_nonfinite_steps": s.skipped}
        with open(f"{R}/metrics.jsonl", "a") as f: f.write(json.dumps(rec) + "\n")
        print("EPOCH", json.dumps(rec), flush=True)
        if not all(torch.isfinite(q).all() for q in pl_module.parameters() if q.requires_grad):
            raise RuntimeError("trainable weights became NaN/inf; stopping so no session time is wasted")
        if s.losses == [] and s.nan_losses:
            raise RuntimeError("every training loss in this epoch was NaN/inf")
        if a.deadline and rec["epoch"] < a.epochs:
            need = 1.25 * max(s.durations[-2:]) + 120
            if time.time() + need > a.deadline:
                print(f"seed {a.seed}: stopping after epoch {rec['epoch']} (next epoch would pass the session deadline)", flush=True)
                trainer.should_stop = True

class Checkpoints(Callback):
    """Own checkpointing, written atomically (temp file + rename), so a kill can never leave a broken file
    and the names stay fixed across sessions: ckpt/last.ckpt after every epoch, ckpt/best.ckpt for the best val mAP."""
    def __init__(s):
        s.best = json.load(open(f"{R}/best.json")) if os.path.exists(f"{R}/best.json") else {"epoch": 0, "val_map": -1.0}
    def on_train_epoch_end(s, trainer, pl_module):  # after this epoch's validation and after EpochLog
        if trainer.sanity_checking: return
        tmp = f"{R}/ckpt/last.ckpt.tmp"; trainer.save_checkpoint(tmp); os.replace(tmp, f"{R}/ckpt/last.ckpt")
        v = float(trainer.callback_metrics.get("val_map", -1.0))
        if v > s.best["val_map"]:
            shutil.copyfile(f"{R}/ckpt/last.ckpt", f"{R}/ckpt/best.ckpt.tmp"); os.replace(f"{R}/ckpt/best.ckpt.tmp", f"{R}/ckpt/best.ckpt")
            s.best = {"epoch": trainer.current_epoch + 1, "val_map": v}; write_json(f"{R}/best.json", s.best)

log = EpochLog(); ckpt = Checkpoints()
trainer = pl.Trainer(accelerator="gpu" if cuda else "cpu", devices=1, precision=a.precision, max_epochs=a.epochs,
                     accumulate_grad_batches=a.accum, deterministic=bool(a.deterministic), benchmark=False,
                     logger=False, enable_checkpointing=False, log_every_n_steps=10, num_sanity_val_steps=0, enable_progress_bar=False,
                     callbacks=[DeterministicAugmentation(), log, ckpt],
                     limit_train_batches=a.limit_train or None, limit_val_batches=a.limit_eval or None,
                     limit_test_batches=a.limit_eval or None)

def ckpt_epochs(path):  # completed epochs stored in a Lightning checkpoint, or None if unreadable
    # The epoch field is an int, but a Lightning .ckpt is a pickle of the whole trainer state,
    # so weights_only=True cannot load it. Only a checkpoint this run wrote is passed here.
    try: return int(torch.load(path, map_location="cpu", weights_only=False, mmap=True)["epoch"]) + 1
    except Exception as e: print("checkpoint unreadable, ignoring:", path, e); return None
last = f"{R}/ckpt/last.ckpt"
done_in_ckpt = ckpt_epochs(last) if os.path.exists(last) else None
resume = last if done_in_ckpt else None
if os.path.exists(f"{R}/metrics.jsonl"):  # drop log lines of epochs the checkpoint does not contain (killed in between)
    keep = done_in_ckpt or 0
    lines = [l for l in open(f"{R}/metrics.jsonl") if l.strip() and json.loads(l)["epoch"] <= keep]
    open(f"{R}/metrics.jsonl", "w").writelines(lines)
print(f"seed {a.seed}: {'resume from ' + resume if resume else 'fresh start'} | precision {a.precision} | "
      f"device {torch.cuda.get_device_name(0) if cuda else 'cpu'}", flush=True)
t0 = time.time()
trainer.fit(model, datamodule=dm, ckpt_path=resume)
done_epochs = trainer.current_epoch  # number of completed epochs after fit
write_json(f"{R}/status.json", {"seed": a.seed, "epochs_done": done_epochs, "epochs_target": a.epochs, "utc": now(),
                                "fit_seconds_this_session": round(time.time() - t0, 1)})
if done_epochs < a.epochs:
    print(f"seed {a.seed}: paused at epoch {done_epochs}/{a.epochs}; resumes in the next session", flush=True); sys.exit(3)

best = f"{R}/ckpt/best.ckpt"
if cuda: torch.cuda.reset_peak_memory_stats()
res = trainer.test(model, datamodule=dm, ckpt_path=best, verbose=False)[0]
recs = [json.loads(l) for l in open(f"{R}/metrics.jsonl")]
write_json(f"{R}/result.json", {
    "seed": a.seed, "test_map": float(res["test_map"]), "test_map_50": float(res["test_map_50"]),
    "test_map_75": float(res.get("test_map_75", float("nan"))), "best_epoch": ckpt.best["epoch"],
    "best_val_map": max((r["val_map"] or 0) for r in recs), "epochs": a.epochs,
    "mean_epoch_seconds": round(sum(r["seconds"] for r in recs) / len(recs), 1),
    "peak_vram_GB": max((r["peak_vram_GB"] or 0) for r in recs) if cuda else None,
    "precision": a.precision, "batch": a.batch, "accum": a.accum, "patch": a.patch, "deterministic": bool(a.deterministic),
    "limit_train": a.limit_train, "limit_eval": a.limit_eval, "utc": now()})
if os.path.exists(last): os.remove(last)  # done: keep only the best checkpoint (saves 1.1 GB of output space)
print(f"seed {a.seed}: DONE test mAP {res['test_map']:.4f} (mAP50 {res['test_map_50']:.4f})", flush=True)


In [ ]:
%%writefile /tmp/realrun/orchestrate.py
"""Run all seeds: one worker per GPU, each pulls the next unfinished seed (2 in parallel, the 3rd after).
Restores earlier progress from /kaggle/input (output of the previous notebook version) before starting.
Stops cleanly before the session deadline so Kaggle can save the output.

Usage: python orchestrate.py --work DIR --data DIR [--seeds 46,47,48 --epochs 30 --deadline UNIX_TS
       --devices auto|0,1|cpu --input-root /kaggle/input --min-start-hours 1.0 -- <extra args for train_seed.py>]"""
import os
os.environ["MPLBACKEND"] = "Agg"
import sys, json, glob, time, shutil, argparse, subprocess, threading, queue
p = argparse.ArgumentParser()
p.add_argument("--work", required=True); p.add_argument("--data", required=True)
p.add_argument("--seeds", default="46,47,48"); p.add_argument("--epochs", type=int, default=30)
p.add_argument("--deadline", type=float, default=0); p.add_argument("--devices", default="auto")
p.add_argument("--input-root", default="/kaggle/input"); p.add_argument("--min-start-hours", type=float, default=1.0)
a, extra = p.parse_known_args()
extra = [x for x in extra if x != "--"]
HERE = os.path.dirname(os.path.abspath(__file__)); RUNS = f"{a.work}/runs"; os.makedirs(RUNS, exist_ok=True)
seeds = [int(s) for s in a.seeds.split(",")]
tag = os.path.basename(os.path.normpath(a.work))

def progress(run_dir):
    if os.path.exists(f"{run_dir}/result.json"): return 10**6
    try: return len({json.loads(l)["epoch"] for l in open(f"{run_dir}/metrics.jsonl") if l.strip()})
    except FileNotFoundError: return -1

# 1) restore from the previous version's output, if it is attached as input and further along
for s in seeds:
    mine = f"{RUNS}/seed{s}"
    cands = [c for c in glob.glob(f"{a.input_root}/**/{tag}/runs/seed{s}", recursive=True) if os.path.isdir(c)]
    if not cands: continue
    best = max(cands, key=progress)
    if progress(best) > progress(mine):
        if os.path.exists(mine): shutil.rmtree(mine)
        shutil.copytree(best, mine)
        print(f"restored seed {s} from {best} (progress {progress(best) if progress(best) < 10**6 else 'done'})", flush=True)
for f in ["manifest.json"]:  # keep the earliest manifest so all sessions document the same selection
    old = glob.glob(f"{a.input_root}/**/{tag}/{f}", recursive=True)
    if old and not os.path.exists(f"{a.work}/{f}"): shutil.copy(old[0], f"{a.work}/{f}")

# 2) devices
if a.devices == "auto":
    try:
        n = len([l for l in subprocess.run(["nvidia-smi", "-L"], capture_output=True, text=True).stdout.splitlines() if l.startswith("GPU")])
    except FileNotFoundError:
        n = 0
    devices = [str(i) for i in range(n)] or ["cpu"]
else:
    devices = a.devices.split(",")
todo = queue.Queue()
for s in seeds:
    if not os.path.exists(f"{RUNS}/seed{s}/result.json"): todo.put(s)
print(f"devices {devices} | open seeds {list(todo.queue)} | deadline in "
      f"{(a.deadline - time.time())/3600:.2f} h" if a.deadline else "no deadline", flush=True)
D = a.data
base = [sys.executable, f"{HERE}/train_seed.py", "--config", f"{D}/model/WAC_config.yaml", "--backbone-dir", f"{D}/model/backbone",
        "--data-dir", f"{D}/hf", "--ann-dir", f"{D}/ann", "--epochs", str(a.epochs)] + (["--deadline", str(a.deadline)] if a.deadline else []) + extra

def worker(dev):
    while True:
        if a.deadline and time.time() + a.min_start_hours * 3600 > a.deadline:
            print(f"[dev {dev}] too little session time left to start another seed", flush=True); return
        try: s = todo.get_nowait()
        except queue.Empty: return
        run = f"{RUNS}/seed{s}"; os.makedirs(run, exist_ok=True)
        env = {**os.environ, "MPLBACKEND": "Agg", "CUDA_VISIBLE_DEVICES": "" if dev == "cpu" else dev}
        logf = open(f"{run}/log.txt", "a"); logf.write(f"\n===== session start {time.strftime('%Y-%m-%d %H:%M:%S')} on dev {dev}\n"); logf.flush()
        print(f"[dev {dev}] seed {s} started", flush=True)
        proc = subprocess.Popen(base + ["--seed", str(s), "--run-dir", run], env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
        for line in proc.stdout:
            logf.write(line)
            if line.startswith(("EPOCH", "seed ")) or "Error" in line or "out of memory" in line: print(f"[dev {dev}] {line.rstrip()}", flush=True)
        rc = proc.wait(); logf.close()
        print(f"[dev {dev}] seed {s} finished with code {rc} ({'done' if rc == 0 else 'paused' if rc == 3 else 'ERROR, see ' + run + '/log.txt'})", flush=True)
        if rc not in (0, 3):
            with open(f"{run}/log.txt") as f: print("".join(f.readlines()[-25:]), flush=True)
        if rc == 3: return  # paused because of the deadline: this device is done for this session

threads = [threading.Thread(target=worker, args=(d,)) for d in devices]
for t in threads: t.start()
for t in threads: t.join()
subprocess.run([sys.executable, f"{HERE}/summarize.py", a.work, "--epochs", str(a.epochs), "--seeds", a.seeds])


In [ ]:
%%writefile /tmp/realrun/summarize.py
"""Summary over all seeds: per-seed test mAP, mean, spread, comparison with NASA's published 0.2581 +- 0.0017.
Usage: python summarize.py WORK_DIR [--epochs 30 --seeds 46,47,48]"""
import os, sys, json, glob, statistics as st, argparse
p = argparse.ArgumentParser(); p.add_argument("work"); p.add_argument("--epochs", type=int, default=30); p.add_argument("--seeds", default="46,47,48")
a = p.parse_args()
NASA = {"test_map": 0.2581, "seed_spread": 0.0017, "test_map_50": 0.6183, "seeds": 5, "source":
        "Model card nasa-ibm-ai4science/Crater-Detection-NASA-IBM-Lunar-Foundation-Model, WAC, 100% data, ps8 LoRA"}
TOL = 0.01
per = {}
for s in [int(x) for x in a.seeds.split(",")]:
    R = f"{a.work}/runs/seed{s}"
    recs = list({r["epoch"]: r for r in (json.loads(l) for l in open(f"{R}/metrics.jsonl") if l.strip())}.values()) if os.path.exists(f"{R}/metrics.jsonl") else []
    entry = {"epochs_done": len(recs), "epochs_target": a.epochs,
             "last_epoch": recs[-1] if recs else None,
             "best_val_map_so_far": max((r["val_map"] or 0) for r in recs) if recs else None}
    if os.path.exists(f"{R}/result.json"):
        entry.update(json.load(open(f"{R}/result.json"))); entry["status"] = "done"
    else:
        entry["status"] = "running/paused" if recs else "not started"
    per[f"seed{s}"] = entry
done = [v for v in per.values() if v["status"] == "done"]
out = {"status": "complete" if len(done) == len(per) else f"{len(done)} of {len(per)} seeds done", "per_seed": per}
if done:
    maps = [v["test_map"] for v in done]; m50 = [v["test_map_50"] for v in done]
    mean = st.mean(maps)
    out["test_map_mean"] = round(mean, 4)
    out["test_map_std"] = round(st.stdev(maps), 4) if len(maps) > 1 else None
    out["test_map_min_max"] = [round(min(maps), 4), round(max(maps), 4)]
    out["test_map_50_mean"] = round(st.mean(m50), 4)
    out["nasa_reference"] = NASA
    out["difference_to_nasa"] = round(mean - NASA["test_map"], 4)
    out["reproduced_within_0.01"] = bool(abs(mean - NASA["test_map"]) <= TOL) if len(done) == len(per) else "pending"
out["setup_differences_to_disclose"] = [
    f"{a.epochs} epochs fixed (model card: up to 100 with early stopping, patience 15)",
    "hyperparameters from released WAC_config.yaml: lr 1e-4, layer decay 1.0, weight decay 0 (model card text: 5e-5, 0.65, 0.05-0.1)",
    "batch 1 x gradient accumulation 4 (NASA: batch 4)",
    "fp16 mixed precision (16-mixed) on Tesla T4; NASA config: fp32",
    "deterministic=False (torchvision roi_align fallback needs too much memory); runs not bit-identical",
    f"{len(per)} seeds (NASA: 5); test set 100 tiles, so differences below the seed spread are not a ranking",
]
json.dump(out, open(f"{a.work}/summary.json", "w"), indent=1)
print(json.dumps(out, indent=1))


In [ ]:
# 3) GPUs prüfen und Python-3.12-Umgebung bauen (ca. 3-5 min)
def sh(cmd): return subprocess.run(cmd, capture_output=True, text=True, env=ENV)
gpu = sh(["nvidia-smi", "--query-gpu=index,name,memory.total,driver_version", "--format=csv"])
assert gpu.returncode == 0, "Keine GPU: rechts Settings -> Accelerator -> GPU T4 x2"
print(gpu.stdout)
r = sh(["bash", f"{CODE}/setup_env.sh"]); print(r.stdout[-1500:]); assert r.returncode == 0, r.stderr[-3000:]
CHECK = "import torch; print('GPUs:', torch.cuda.device_count(), '| torch', torch.__version__, '| CUDA-Build', torch.version.cuda)"
def ngpu():
    r = sh([PY, "-c", CHECK]); print((r.stdout + r.stderr).strip()[-800:])
    try: return int(r.stdout.split("GPUs:")[1].split("|")[0])
    except Exception: return 0
n = ngpu()
if n == 0:
    print("PyTorch sieht keine GPU -> CUDA-Räder neu installieren ...")
    r = sh(["bash", f"{CODE}/setup_env.sh", "--force-torch"]); print(r.stdout[-1500:]); n = ngpu()
assert n > 0, "PyTorch sieht keine GPU. Bitte die Ausgabe dieser Zelle prüfen."
if n < 2: print("Hinweis: nur 1 GPU sichtbar. Läuft, aber die Seeds laufen dann nacheinander (langsamer). Für 2 parallel: GPU T4 x2 wählen.")
print("GPU bereit.")

In [ ]:
# 4) Daten laden (fest versioniert). Smoke: 40/10/10 Kacheln. Full: alle 800/100/100 (~4 GB, ca. 5-10 min)
subset = [] if MODE == "full" else ["--subset", "40,10,10"]
r = subprocess.run([PY, f"{CODE}/fetch_data.py", DATA, f"{WORK}/manifest.json", "--seeds", SEEDS] + subset, env=ENV,
                   stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
print("\n".join(l for l in r.stdout.splitlines() if "Fetching" not in l and "Warning" not in l)[-2000:])
if r.returncode != 0:
    out = r.stdout
    if "HTTP 429" in out or "Too Many Requests" in out or any(f"HTTP {c}" in out for c in (500, 502, 503, 504)):
        raise RuntimeError("Download fehlgeschlagen: Hugging Face war auch nach 10 Versuchen überlastet (HTTP 429/5xx). "
                           "Internet ist an. Einfach später erneut 'Save & Run All' starten.")
    if "NameResolution" in out or "Failed to resolve" in out or "Temporary failure in name resolution" in out:
        raise RuntimeError("Download fehlgeschlagen: kein Internet. Rechts unter Settings 'Internet' einschalten.")
    raise RuntimeError("Download fehlgeschlagen (Details oben). Bitte die Ausgabe dieser Zelle prüfen.")

In [ ]:
# 5) Training. Full: Seeds 46 und 47 parallel (je eine GPU), 48 danach. Fortschritt erscheint hier pro Epoche.
EPOCHS = 30 if MODE == "full" else 1
train_args = ["--precision", "16-mixed", "--batch", "1", "--accum", "4", "--patch", "8", "--deterministic", "0", "--num-workers", "2"]
if MODE != "full": train_args += ["--limit-train", "40", "--limit-eval", "10"]
cmd = [PY, f"{CODE}/orchestrate.py", "--work", WORK, "--data", DATA, "--seeds", SEEDS, "--epochs", str(EPOCHS),
       "--deadline", str(DEADLINE), "--input-root", "/kaggle/input", "--min-start-hours", "1.0", "--"] + train_args
p = subprocess.Popen(cmd, env=ENV, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
for line in p.stdout:
    if not line.startswith((" ", "{", "}")): print(line.rstrip(), flush=True)   # summary JSON comes in the next cell
print("Orchestrator-Exitcode:", p.wait())

In [ ]:
# 6) Zusammenfassung
s = json.load(open(f"{WORK}/summary.json"))
print(json.dumps(s, indent=1))
if MODE != "full":
    ok = all(v["status"] == "done" and v.get("last_epoch") and v["last_epoch"]["train_loss"] is not None
             and v["last_epoch"]["nan_loss_batches"] == 0 for v in s["per_seed"].values())
    ep = [v["last_epoch"] for v in s["per_seed"].values() if v.get("last_epoch")]
    if ep:
        sec = max(e["seconds"] for e in ep)   # 40 Trainings- + 10 Validierungs-Kacheln, beide GPUs parallel
        print(f"Grobe Hochrechnung: ~{sec / 50 * 900 / 60:.0f} min pro Epoche im echten Lauf "
              f"(~{sec / 50 * 900 * 30 / 3600:.1f} h für 30 Epochen pro Seed)")
    print("SMOKE OK: jetzt MODE = \"full\" setzen und 'Save & Run All' starten." if ok else "SMOKE FEHLER: bitte die Ausgabe dieser Zelle prüfen.")
elif s["status"] == "complete":
    print(f"FERTIG: Mittelwert {s['test_map_mean']} (NASA 0.2581), Abweichung {s['difference_to_nasa']}, "
          f"innerhalb ±0,01: {s['reproduced_within_0.01']}")
else:
    print("NOCH NICHT FERTIG (" + s["status"] + "). Fortsetzen: Input -> Add Input -> dieses Notebook (neueste Version), dann wieder 'Save & Run All'.")